In [ ]:
import os
from pathlib import Path

# Create required directories
for d in ["encoder", "encoder/checkpoints", "encoder/plots", "configs"]:
    Path(d).mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = ["BTC_USDT_aligned.parquet", "ETH_USDT_aligned.parquet",
                   "EURUSD_aligned.parquet", "XAUUSD_aligned.parquet"]

# ── Auto-discover the dataset — don't hardcode the slug ─────────────────────
# Kaggle sometimes nests the uploaded folder one level deeper than expected, and
# the exact input slug depends on what you named the dataset when you created it
# (it may not be exactly "marta-processed-data-v3"). Instead of guessing a fixed
# path and crashing opaquely, search every dataset actually mounted under
# /kaggle/input/ for a directory holding all 4 expected files.
KAGGLE_INPUT = Path("/kaggle/input")
print("Datasets attached under /kaggle/input/:")
if KAGGLE_INPUT.exists():
    for d in sorted(KAGGLE_INPUT.iterdir()):
        print(f"  {d}")
else:
    print("  /kaggle/input does not exist — are you running this outside Kaggle?")

DATA_DIR = None
if KAGGLE_INPUT.exists():
    for candidate_dir in [KAGGLE_INPUT] + sorted(KAGGLE_INPUT.rglob("*")):
        if not candidate_dir.is_dir():
            continue
        if all((candidate_dir / f).exists() for f in EXPECTED_FILES):
            DATA_DIR = candidate_dir
            break

if DATA_DIR is not None:
    print(f"\n✓ Found all 4 aligned parquets under: {DATA_DIR}")
else:
    print(
        "\n✗ Could not find BTC_USDT_aligned.parquet + co. anywhere under /kaggle/input/.\n"
        "  Likely causes:\n"
        "    1. The marta-processed-data-v3 dataset isn't attached to this notebook yet\n"
        "       — click 'Add Input' (right sidebar) and attach it.\n"
        "    2. It's still uploading/processing on Kaggle's side.\n"
        "    3. It was created under a different name — check the listing above and\n"
        "       compare to whatever you actually typed when creating the dataset.\n"
        "  Fix the attachment, then re-run this cell before continuing."
    )

# Print whatever parquet files exist anywhere under /kaggle/input/, for extra context
print("\nAll .parquet files found under /kaggle/input/ (recursive):")
if KAGGLE_INPUT.exists():
    for f in sorted(KAGGLE_INPUT.rglob("*.parquet")):
        print(f"  {f}")

In [ ]:
# Directory setup + data path verification (uses DATA_DIR resolved by the cell above)
import os
from pathlib import Path

# Create required directories
for d in ["encoder", "encoder/checkpoints", "encoder/plots", "configs", "utils"]:
    Path(d).mkdir(parents=True, exist_ok=True)

assert DATA_DIR is not None, (
    "DATA_DIR is None — the dataset wasn't found under /kaggle/input/ in the cell "
    "above. Attach marta-processed-data-v3 to this notebook and re-run from the top."
)
expected = EXPECTED_FILES

for f in expected:
    path = DATA_DIR / f
    assert path.exists(), f"Missing: {path}"
    print(f"✓ {f}")

print(f"\nAll data files found under {DATA_DIR}.")

In [ ]:
# Leakage guard — 2025 must NEVER reach this dataset. If this assert fails, STOP:
# it means the wrong Kaggle dataset was uploaded and the test set would be
# compromised for every downstream layer (HMM, RL) built on this checkpoint.
import pandas as pd

CUTOFF = pd.Timestamp("2024-12-31 23:59:59", tz="UTC")
for f in expected:
    df = pd.read_parquet(DATA_DIR / f, engine="pyarrow")
    assert df.index[-1] <= CUTOFF, (
        f"{f} contains post-2024 rows (ends {df.index[-1]}) — test-set leakage, STOP"
    )
    print(f"✓ {f}: {len(df)} rows, {df.index[0].date()} -> {df.index[-1].date()}")

print("\nOK — no 2025 rows on Kaggle, split dates correct")

In [ ]:
%%writefile utils/__init__.py
from utils.logger import logger

__all__ = ["logger"]

In [ ]:
%%writefile utils/logger.py
from __future__ import annotations
import logging
from datetime import datetime
from rich.console import Console
from rich.logging import RichHandler

console = Console()

class MartaLogger:
    def __init__(self) -> None:
        logging.basicConfig(
            level=logging.INFO,
            format="%(message)s",
            handlers=[RichHandler(console=console, rich_tracebacks=True)]
        )
        self._log = logging.getLogger("marta")

    def info(self, msg: str) -> None:
        self._log.info(msg)

    def warning(self, msg: str) -> None:
        self._log.warning(msg)

    def error(self, msg: str) -> None:
        self._log.error(msg)

    def success(self, msg: str) -> None:
        self._log.info(f"[green]{msg}[/green]", extra={"markup": True})

logger = MartaLogger()

In [ ]:
import subprocess
subprocess.run(["pip", "install", "pytorch-forecasting", 
                "mlflow", "ta", "hmmlearn", "-q"])

In [ ]:
%%writefile encoder/augment.py
"""
Contrastive augmentation functions for regime windows.

All functions are pure, stateless, and thread-safe: each call creates a local
np.random.Generator seeded from OS entropy — safe for multi-worker DataLoaders.
"""

from __future__ import annotations

import numpy as np


def amplitude_scale(window: np.ndarray, p: float = 0.5) -> np.ndarray:
    """Multiply all values by a scalar drawn from Uniform(0.85, 1.15)."""
    rng = np.random.default_rng()
    if rng.random() >= p:
        return window.copy()
    scale = rng.uniform(0.85, 1.15)
    return (window * scale).astype(np.float32)


def add_jitter(window: np.ndarray, p: float = 0.5, sigma: float = 0.03) -> np.ndarray:
    """Add i.i.d. Gaussian noise with std=sigma to every element."""
    rng = np.random.default_rng()
    if rng.random() >= p:
        return window.copy()
    noise = rng.normal(0.0, sigma, window.shape).astype(np.float32)
    return (window + noise).astype(np.float32)


def channel_mask(window: np.ndarray, p: float = 0.5, n_mask: int = 2) -> np.ndarray:
    """Zero out n_mask randomly chosen feature columns for the entire window."""
    rng = np.random.default_rng()
    if rng.random() >= p:
        return window.copy()
    result = window.copy()
    n_features = window.shape[1]
    mask_cols = rng.choice(n_features, size=n_mask, replace=False)
    result[:, mask_cols] = 0.0
    return result


def time_warp(window: np.ndarray, p: float = 0.3, max_shift: float = 5.0) -> np.ndarray:
    """
    Elastic distortion of the time axis via a smooth sin-wave perturbation.

    Strategy: treat the original data as sampled on a regular grid and resample
    it as if the grid were warped by a sin wave with random phase and amplitude.
    np.interp(src, warp, data) reads from the warped positions and maps them
    back onto the regular output grid, preserving shape (T, F) exactly.
    """
    rng = np.random.default_rng()
    if rng.random() >= p:
        return window.copy()
    T, F = window.shape
    src = np.linspace(0, T - 1, T)
    phase = rng.uniform(0, 2 * np.pi)
    amplitude = rng.uniform(0, max_shift)
    # warp is analytically monotone for amplitude <= max_shift and T=60
    # (min derivative = 1 - amplitude * 2π/T = 1 - 5 * 2π/60 ≈ 0.48 > 0)
    warp = src + amplitude * np.sin(src * 2.0 * np.pi / T + phase)
    warp = np.clip(warp, 0, T - 1)
    result = np.empty_like(window)
    for f in range(F):
        result[:, f] = np.interp(src, warp, window[:, f])
    return result.astype(np.float32)


def augment_window(window: np.ndarray) -> np.ndarray:
    """Apply all four augmentations sequentially and independently."""
    x = amplitude_scale(window)
    x = add_jitter(x)
    x = channel_mask(x)
    x = time_warp(x)
    return x


In [ ]:
%%writefile encoder/dataset.py

from __future__ import annotations

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from encoder.augment import augment_window


class RegimeContrastiveDataset(Dataset):
    """
    Sliding-window contrastive dataset over a single asset's aligned time series.

    Returns (anchor, positive) pairs where both views are independently augmented
    copies of the same raw window (SimCLR design — no raw windows are returned).
    """

    def __init__(
        self,
        df: pd.DataFrame,
        window_size: int = 60,
        step: int = 1,
    ) -> None:
        # Convert once; __getitem__ never touches the DataFrame again
        self.data: np.ndarray = df.values.astype(np.float32)
        self.window_size = window_size
        self.indices: list[int] = list(range(0, len(df) - window_size, step))

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        start = self.indices[idx]
        window = self.data[start : start + self.window_size]  # (60, 13)
        anchor = torch.from_numpy(augment_window(window))
        positive = torch.from_numpy(augment_window(window))
        return anchor, positive


In [ ]:
%%writefile encoder/model.py
"""
TFT-inspired regime encoder for MARTA.

Architecture replicates the encoder portion of a Temporal Fusion Transformer:
  VSN (feature gating) → 2-layer LSTM → mean pool → gated residual → projection head.

Month 2 refinement note:
  Replace the VSN + LSTM block with pytorch-forecasting's TemporalFusionTransformer
  by passing windows through TFT.predict() and extracting the internal encoder hidden
  states (access via monkey-patching TFT._get_encoder_output or a forward hook on
  the lstm_encoder sub-module). The projection head and gated residual remain unchanged.
"""

from __future__ import annotations

import torch
import torch.nn as nn


class TFTRegimeEncoder(nn.Module):
    def __init__(
        self,
        input_dim: int = 14,
        hidden_dim: int = 128,
        proj_dim: int = 64,
        n_lstm_layers: int = 2,
    ) -> None:
        super().__init__()

        # Variable Selection Network: learned per-timestep feature importance
        self.vsn_linear = nn.Linear(input_dim, input_dim)

        # Sequence encoder (mirrors TFT's internal LSTM encoder)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=n_lstm_layers,
            batch_first=True,
            dropout=0.1 if n_lstm_layers > 1 else 0.0,
        )

        # Gated residual connection applied after mean pooling
        self.gate = nn.Linear(hidden_dim, hidden_dim)

        # Projection head — discarded after pretraining; only used by NTXentLoss
        self.projector = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, proj_dim),
        )

    def _vsn(self, x: torch.Tensor) -> torch.Tensor:
        """Apply VSN: per-timestep softmax gating over features."""
        weights = torch.softmax(self.vsn_linear(x), dim=-1)  # (B, T, 14)
        return x * weights

    def _encode(self, x: torch.Tensor) -> torch.Tensor:
        """Shared encoder path used by both forward() and encode()."""
        x_weighted = self._vsn(x)                        # (B, T, 14)
        lstm_out, _ = self.lstm(x_weighted)               # (B, T, 128)
        h = lstm_out.mean(dim=1)                          # (B, 128) — robust to regime length
        gate = torch.sigmoid(self.gate(h))                # (B, 128)
        return h * gate                                   # (B, 128)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return (embedding, projection) — projection used only during contrastive training."""
        embedding = self._encode(x)                       # (B, 128)
        projection = self.projector(embedding)            # (B, 64)
        return embedding, projection

    def get_vsn_weights(self, x: torch.Tensor) -> torch.Tensor:
        """Per-timestep VSN feature importance weights (B, T, 14).

        Passed to the Technical Agent in Month 2 as the attention map.
        """
        return torch.softmax(self.vsn_linear(x), dim=-1)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Return only the 128-dim embedding — used for inference after pretraining."""
        return self._encode(x)


In [ ]:
%%writefile encoder/loss.py

from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def infonce_loss(
    z_anchor: torch.Tensor,
    z_positive: torch.Tensor,
    temperature: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute InfoNCE loss and top-1 accuracy for a batch of contrastive pairs.

    Returns (loss, top1_accuracy) — both scalar tensors.
    Random-chance baseline: loss = log(batch_size), accuracy = 1/batch_size.
    """
    z_a = F.normalize(z_anchor, dim=1)   # (B, D)
    z_p = F.normalize(z_positive, dim=1) # (B, D)

    batch_size = z_a.shape[0]
    sim_matrix = torch.mm(z_a, z_p.T) / temperature  # (B, B)

    labels = torch.arange(batch_size, device=z_anchor.device)
    loss = F.cross_entropy(sim_matrix, labels)

    with torch.no_grad():
        preds = sim_matrix.argmax(dim=1)
        top1_acc = (preds == labels).float().mean()

    return loss, top1_acc


class NTXentLoss(nn.Module):
    """
    NT-Xent loss with a learnable temperature parameter.

    Temperature is parameterised as exp(-log_temp) so that gradient descent
    can anneal it during training without going negative.
    Initialised to log(1/0.07) ≈ 2.66 → effective starting temperature = 0.07.
    """

    def __init__(self) -> None:
        super().__init__()
        # log(1/0.07) initialisation → temperature = exp(-log_temp) = 0.07
        self.log_temp = nn.Parameter(torch.tensor(math.log(1.0 / 0.07)))

    @property
    def temperature(self) -> torch.Tensor:
        return torch.exp(-self.log_temp)

    def clamp_temperature(self, min_temperature: float, max_temperature: float) -> None:
        """Hard-bound ``log_temp`` in place so ``temperature = exp(-log_temp)`` cannot
        wander outside ``[min_temperature, max_temperature]``.

        Call after each optimizer step. Gradient descent on an unconstrained log_temp was
        observed to drive temperature to 0.0047 (6.4x below the 0.03 floor) during the
        2026-07 Kaggle walk-forward retrain — the convergence check in encoder/train.py
        only warns on this, it doesn't constrain it, so it went to production unnoticed.
        """
        with torch.no_grad():
            # temperature <= max_temperature  <=>  log_temp >= -ln(max_temperature)
            # temperature >= min_temperature  <=>  log_temp <= -ln(min_temperature)
            low = -math.log(max_temperature)
            high = -math.log(min_temperature)
            self.log_temp.clamp_(min=low, max=high)

    def forward(
        self,
        z_anchor: torch.Tensor,
        z_positive: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        z_a = F.normalize(z_anchor, dim=1)
        z_p = F.normalize(z_positive, dim=1)

        batch_size = z_a.shape[0]
        # Divide by self.temperature tensor so gradient flows to log_temp
        sim_matrix = torch.mm(z_a, z_p.T) / self.temperature  # (B, B)

        labels = torch.arange(batch_size, device=z_anchor.device)
        loss = F.cross_entropy(sim_matrix, labels)

        with torch.no_grad():
            preds = sim_matrix.argmax(dim=1)
            top1_acc = (preds == labels).float().mean()

        return loss, top1_acc


In [ ]:
%%writefile encoder/train.py
"""
Week 2 encoder training loop — contrastive pretraining with InfoNCE loss.

Data split: train < train_end, val >= val_start (within actual data range 2018-2021).
All 5 assets are concatenated into a shared ConcatDataset so the encoder learns
a single embedding space across crypto, forex, and commodities.
"""

from __future__ import annotations

from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import ConcatDataset, DataLoader

from encoder.dataset import RegimeContrastiveDataset
from encoder.embed import extract_embeddings
from encoder.loss import NTXentLoss
from encoder.model import TFTRegimeEncoder
from encoder.visualize import plot_training_curves, plot_tsne_embeddings
from utils.logger import logger

_ROOT = Path(__file__).resolve().parent.parent


def _load_asset_split(
    asset: str,
    processed_dir: Path,
    train_end: str,
    val_start: str,
    window_size: int,
    step: int,
) -> tuple[RegimeContrastiveDataset, RegimeContrastiveDataset]:
    df = pd.read_parquet(processed_dir / f"{asset}_aligned.parquet", engine="pyarrow")
    train_end_ts = pd.Timestamp(train_end, tz="UTC")
    val_start_ts = pd.Timestamp(val_start, tz="UTC")

    train_df = df[df.index < train_end_ts]
    val_df = df[df.index >= val_start_ts]

    train_ds = RegimeContrastiveDataset(train_df, window_size=window_size, step=step)
    val_ds = RegimeContrastiveDataset(val_df, window_size=window_size, step=step)
    return train_ds, val_ds


def _run_epoch(
    loader: DataLoader,
    model: TFTRegimeEncoder,
    loss_fn: NTXentLoss,
    optimizer: AdamW | None,
    device: torch.device,
    grad_clip: float,
    train: bool,
    temp_bounds: tuple[float, float] | None = None,
) -> tuple[float, float, bool]:
    """Run one full pass over loader.  Returns (mean_loss, mean_acc, nan_grad_hit)."""
    model.train(train)
    nan_grad_hit = False
    losses: list[float] = []
    accs: list[float] = []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for anchor, positive in loader:
            anchor = anchor.to(device, non_blocking=True)
            positive = positive.to(device, non_blocking=True)

            _, z_a = model(anchor)
            _, z_p = model(positive)
            loss, acc = loss_fn(z_a, z_p)

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                all_params = list(model.parameters()) + list(loss_fn.parameters())
                grad_norm = nn.utils.clip_grad_norm_(all_params, max_norm=grad_clip)
                if torch.isnan(grad_norm):
                    nan_grad_hit = True
                optimizer.step()
                if temp_bounds is not None:
                    loss_fn.clamp_temperature(*temp_bounds)

            losses.append(loss.item())
            accs.append(acc.item())

    return float(np.mean(losses)), float(np.mean(accs)), nan_grad_hit


def train_encoder(config: dict) -> TFTRegimeEncoder:
    """
    Full contrastive pretraining loop.

    Trains TFTRegimeEncoder with NTXentLoss across all 5 aligned asset datasets,
    logs every epoch to MLflow, applies early stopping, saves best checkpoint,
    extracts embeddings, and plots t-SNE — all within the caller's MLflow run.

    Returns the best encoder (lowest val loss).
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Training on {device}")

    # ------------------------------------------------------------------ config
    mcfg = config["model"]
    tcfg = config["training"]
    dcfg = config["data"]
    vcfg = config["visualization"]
    conv = config["convergence"]
    ckpt_cfg = config["checkpoints"]

    window_size: int = tcfg["window_size"]
    step: int = tcfg["step"]
    batch_size: int = tcfg["batch_size"]
    lr: float = tcfg["lr"]
    weight_decay: float = tcfg["weight_decay"]
    max_epochs: int = tcfg["max_epochs"]
    patience: int = tcfg["early_stopping_patience"]
    warmup_epochs: int = tcfg["warmup_epochs"]
    grad_clip: float = tcfg["gradient_clip_max_norm"]
    train_end: str = tcfg["train_end"]
    val_start: str = tcfg["val_start"]

    processed_dir = _ROOT / dcfg["processed_dir"]
    ckpt_dir = _ROOT / ckpt_cfg["dir"]
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    plots_dir = _ROOT / "encoder" / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------------------------------------------- data
    train_datasets: list[RegimeContrastiveDataset] = []
    val_datasets: list[RegimeContrastiveDataset] = []
    for asset in dcfg["assets"]:
        tr_ds, va_ds = _load_asset_split(
            asset, processed_dir, train_end, val_start, window_size, step
        )
        if len(tr_ds) > 0:
            train_datasets.append(tr_ds)
        if len(va_ds) > 0:
            val_datasets.append(va_ds)
        logger.info(f"{asset}: {len(tr_ds)} train windows, {len(va_ds)} val windows")

    pin = device.type == "cuda"
    train_loader = DataLoader(
        ConcatDataset(train_datasets),
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=pin,
    )
    val_loader = DataLoader(
        ConcatDataset(val_datasets),
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=pin,
    )

    # -------------------------------------------------------------- model
    model = TFTRegimeEncoder(
        input_dim=mcfg["input_dim"],
        hidden_dim=mcfg["hidden_dim"],
        proj_dim=mcfg["proj_dim"],
        n_lstm_layers=mcfg["n_lstm_layers"],
    ).to(device)
    loss_fn = NTXentLoss().to(device)

    all_params = list(model.parameters()) + list(loss_fn.parameters())
    optimizer = AdamW(all_params, lr=lr, weight_decay=weight_decay)

    # Linear warmup then cosine annealing
    warmup_sched = LinearLR(
        optimizer, start_factor=1e-8, end_factor=1.0, total_iters=warmup_epochs
    )
    cosine_sched = CosineAnnealingLR(
        optimizer, T_max=max(max_epochs - warmup_epochs, 1)
    )
    scheduler = SequentialLR(
        optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_epochs]
    )

    # -------------------------------------------------------------- training
    best_val_loss = float("inf")
    best_epoch = 0
    best_val_acc = 0.0
    patience_counter = 0
    nan_grad_detected = False
    stopping_epoch = max_epochs

    temp_bounds = (conv["min_temperature"], conv["max_temperature"])

    for epoch in range(1, max_epochs + 1):
        train_loss, train_acc, nan_hit = _run_epoch(
            train_loader, model, loss_fn, optimizer, device, grad_clip, train=True,
            temp_bounds=temp_bounds,
        )
        if nan_hit:
            nan_grad_detected = True

        val_loss, val_acc, _ = _run_epoch(
            val_loader, model, loss_fn, None, device, grad_clip, train=False
        )

        scheduler.step()

        current_temp = loss_fn.temperature.item()
        current_lr = optimizer.param_groups[0]["lr"]

        mlflow.log_metrics(
            {
                "train_loss": train_loss,
                "train_top1_acc": train_acc,
                "val_loss": val_loss,
                "val_top1_acc": val_acc,
                "temperature": current_temp,
                "learning_rate": current_lr,
            },
            step=epoch,
        )

        logger.info(
            f"Epoch {epoch:03d} | "
            f"train {train_loss:.4f} ({train_acc:.3f}) | "
            f"val {val_loss:.4f} ({val_acc:.3f}) | "
            f"T={current_temp:.4f} | lr={current_lr:.2e}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_epoch = epoch
            patience_counter = 0
            ckpt_data = {
                "epoch": epoch,
                "val_loss": val_loss,
                "val_top1_acc": val_acc,
                "model_state": model.state_dict(),
                "loss_fn_state": loss_fn.state_dict(),
            }
            # Save to working dir (survives session interruption)
            torch.save(ckpt_data, Path("/kaggle/working/best_encoder.pt"))
            # Save to local checkpoints dir as well
            torch.save(ckpt_data, ckpt_dir / ckpt_cfg["best"])
        else:
            patience_counter += 1
            if patience_counter >= patience:
                stopping_epoch = epoch
                logger.info(f"Early stopping triggered at epoch {epoch}")
                mlflow.log_param("early_stopping_epoch", epoch)
                break

    # Save final checkpoint and log both
    torch.save(
        {
            "epoch": epoch,
            "val_loss": val_loss,
            "val_top1_acc": val_acc,
            "model_state": model.state_dict(),
            "loss_fn_state": loss_fn.state_dict(),
        },
        ckpt_dir / ckpt_cfg["final"],
    )
    mlflow.log_artifact(str(ckpt_dir / ckpt_cfg["best"]))
    mlflow.log_artifact(str(ckpt_dir / ckpt_cfg["final"]))

    mlflow.log_params(
        {
            "best_epoch": best_epoch,
            "best_val_loss": round(best_val_loss, 6),
            "stopping_epoch": stopping_epoch,
        }
    )

    # -------------------------------------------------------------- load best for inference
    best_ckpt = torch.load(
        ckpt_dir / ckpt_cfg["best"], map_location=device, weights_only=True
    )
    model.load_state_dict(best_ckpt["model_state"])
    loss_fn.load_state_dict(best_ckpt["loss_fn_state"])

    # -------------------------------------------------------------- convergence checks
    checks = _run_convergence_checks(
        best_val_loss=best_val_loss,
        best_val_acc=best_val_acc,
        temperature=loss_fn.temperature.item(),
        nan_grad_detected=nan_grad_detected,
        thresholds=conv,
    )

    # Embed the convergence-check results and thresholds INTO the checkpoint file itself —
    # not just MLflow. When training runs on Kaggle (external, no local MLflow), the
    # checkpoint .pt is the only artifact that reliably gets downloaded; if convergence
    # metrics only live in a throwaway remote MLflow run, they're lost the moment the
    # Kaggle session ends. This is exactly how the 2026-07 Kaggle retrain's collapsed
    # temperature (0.0047, 6.4x below the 0.03 floor) went to production unnoticed.
    provenance_paths = [
        Path("/kaggle/working/best_encoder.pt"),
        ckpt_dir / ckpt_cfg["best"],
        ckpt_dir / ckpt_cfg["final"],
    ]
    for path in provenance_paths:
        if not path.exists():
            continue
        saved = torch.load(path, map_location="cpu", weights_only=True)
        saved["convergence_checks"] = checks
        saved["convergence_thresholds"] = dict(conv)
        torch.save(saved, path)

    # -------------------------------------------------------------- post-training artefacts
    run_id = mlflow.active_run().info.run_id

    logger.info("Extracting embeddings …")
    embedding_matrix, timestamps = extract_embeddings(model, config)

    logger.info("Rendering t-SNE …")
    tsne_path = plots_dir / "tsne_embeddings.png"
    try:
        plot_tsne_embeddings(
            embedding_matrix,
            timestamps,
            tsne_path,
            n_samples=vcfg["tsne_n_samples"],
        )
        mlflow.log_metric("tsne_saved", 1)
    except Exception as exc:  # noqa: BLE001
        logger.error(f"t-SNE failed: {exc}")
        mlflow.log_metric("tsne_saved", 0)

    logger.info("Rendering training curves …")
    curves_path = plots_dir / "training_curves.png"
    try:
        plot_training_curves(run_id, curves_path)
    except Exception as exc:  # noqa: BLE001
        logger.error(f"Training curves plot failed: {exc}")

    return model


def _run_convergence_checks(
    best_val_loss: float,
    best_val_acc: float,
    temperature: float,
    nan_grad_detected: bool,
    thresholds: dict,
) -> dict[str, bool]:
    """Log all convergence checks to MLflow, surface CRITICAL failures, and return the
    pass/fail dict so the caller can embed it in the checkpoint file itself."""
    checks: dict[str, bool] = {
        "val_loss_below_threshold": best_val_loss < thresholds["max_val_loss"],
        "top1_acc_above_threshold": best_val_acc > thresholds["min_top1_acc"],
        "temperature_in_range": thresholds["min_temperature"] <= temperature <= thresholds["max_temperature"],
        "no_nan_gradients": not nan_grad_detected,
    }
    severity = {
        "val_loss_below_threshold": "CRITICAL",
        "top1_acc_above_threshold": "CRITICAL",
        "temperature_in_range": "WARNING",
        "no_nan_gradients": "CRITICAL",
    }
    diagnostics = {
        "val_loss_below_threshold": (
            f"Val loss {best_val_loss:.4f} >= {thresholds['max_val_loss']} "
            f"(random baseline ≈ {__import__('math').log(512):.2f}). "
            "Check: batch size too small, learning rate too high, or augmentation too aggressive."
        ),
        "top1_acc_above_threshold": (
            f"Top-1 acc {best_val_acc:.4f} <= {thresholds['min_top1_acc']}. "
            "Check: projection head collapse, temperature too large, or data too uniform."
        ),
        "temperature_in_range": (
            f"Temperature {temperature:.4f} outside [{thresholds['min_temperature']}, "
            f"{thresholds['max_temperature']}]. Usually harmless but monitor for instability."
        ),
        "no_nan_gradients": (
            "NaN gradients detected during training. "
            "Check: reduce lr, add gradient clipping, or inspect input normalisation."
        ),
    }

    for name, passed in checks.items():
        mlflow.log_metric(f"check_{name}", int(passed))
        level = severity[name]
        if not passed:
            msg = f"[{level}] Convergence check FAILED — {name}: {diagnostics[name]}"
            if level == "CRITICAL":
                logger.error(msg)
            else:
                logger.warning(msg)
        else:
            logger.success(f"[{level}] Convergence check PASSED — {name}")

    return checks


In [ ]:
%%writefile encoder/embed.py
"""
Post-training embedding extraction.

Slides a 60-candle window over each asset's full aligned dataset (no train/val split)
and runs model.encode() in batches to produce a unified embedding matrix.
"""

from __future__ import annotations

from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import torch

from encoder.model import TFTRegimeEncoder

_ROOT = Path(__file__).resolve().parent.parent


def extract_embeddings(
    model: TFTRegimeEncoder,
    config: dict,
) -> tuple[np.ndarray, list]:
    """
    Embed every sliding window for all assets and persist results to data/processed/.

    Returns (embedding_matrix, timestamps) where:
      embedding_matrix: (n_total_windows, 128)
      timestamps:       list of end-of-window timestamps (length n_total_windows)
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)

    window_size: int = config["training"]["window_size"]
    assets: list[str] = config["data"]["assets"]
    processed_dir = _ROOT / config["data"]["processed_dir"]
    # Read path (processed_dir) may be a read-only mount (e.g. /kaggle/input).
    # Generated artifacts go to output_dir, which defaults to processed_dir but
    # can be overridden to a writable location (e.g. /kaggle/working/data/processed).
    out_dir = Path(config["data"].get("output_dir") or processed_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    batch_size = 512

    all_embeddings: list[np.ndarray] = []
    all_timestamps: list = []
    asset_map_rows: list[dict] = []
    total_idx = 0

    for asset in assets:
        df = pd.read_parquet(
            processed_dir / f"{asset}_aligned.parquet", engine="pyarrow"
        )
        data = df.values.astype(np.float32)           # (T, 14)
        timestamps = df.index.tolist()

        n_windows = len(df) - window_size
        if n_windows <= 0:
            continue

        # Collect raw windows and their end timestamps
        windows: list[np.ndarray] = []
        win_timestamps: list = []
        for start in range(n_windows):
            windows.append(data[start : start + window_size])
            win_timestamps.append(timestamps[start + window_size - 1])

        # Batch encode
        asset_embeddings: list[np.ndarray] = []
        with torch.no_grad():
            for i in range(0, len(windows), batch_size):
                batch = np.stack(windows[i : i + batch_size])
                tensor = torch.from_numpy(batch).to(device)
                emb = model.encode(tensor)             # (B, 128)
                asset_embeddings.append(emb.cpu().numpy())

        asset_emb_arr = np.concatenate(asset_embeddings, axis=0)  # (n_windows, 128)
        all_embeddings.append(asset_emb_arr)
        all_timestamps.extend(win_timestamps)

        asset_map_rows.append(
            {
                "start_idx": total_idx,
                "end_idx": total_idx + n_windows,
                "asset_name": asset,
            }
        )
        total_idx += n_windows

    embedding_matrix = np.concatenate(all_embeddings, axis=0)  # (N, 128)

    # Persist (to writable out_dir, not the possibly read-only read path)
    np.save(out_dir / "embeddings.npy", embedding_matrix)
    pd.DataFrame({"timestamp": all_timestamps}).to_parquet(
        out_dir / "embedding_timestamps.parquet", engine="pyarrow"
    )
    pd.DataFrame(asset_map_rows).to_parquet(
        out_dir / "embedding_asset_map.parquet", engine="pyarrow"
    )

    # MLflow logging
    mlflow.log_metric("embedding_rows", embedding_matrix.shape[0])
    mlflow.log_metric("embedding_dim", embedding_matrix.shape[1])
    for row in asset_map_rows:
        n = row["end_idx"] - row["start_idx"]
        mlflow.log_metric(f"windows_{row['asset_name']}", n)

    return embedding_matrix, all_timestamps


In [ ]:
%%writefile encoder/visualize.py
"""
Training visualisations: t-SNE embedding plot and training-curve dashboard.
"""

from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE

_ROOT = Path(__file__).resolve().parent.parent
_ASSET_COLORS = {
    "BTC_USDT": "#F7931A",
    "ETH_USDT": "#627EEA",
    "EURUSD": "#2E86AB",
    "XAUUSD": "#FFD700",
}


def plot_tsne_embeddings(
    embeddings: np.ndarray,
    timestamps: list,
    save_path: Path,
    n_samples: int = 3000,
) -> None:
    """
    Two-panel t-SNE plot:
      Left  — coloured by chronological order (viridis) to reveal temporal drift.
      Right — coloured by asset.
    Subsamples n_samples rows with fixed seed=42 for reproducibility.
    Saves PNG to save_path and logs it as an MLflow artifact.
    """
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    # Load asset map for per-asset colouring
    asset_map_path = _ROOT / "data" / "processed" / "embedding_asset_map.parquet"
    asset_map = pd.read_parquet(asset_map_path, engine="pyarrow")
    n_total = embeddings.shape[0]
    asset_labels = np.empty(n_total, dtype=object)
    for _, row in asset_map.iterrows():
        asset_labels[int(row["start_idx"]) : int(row["end_idx"])] = row["asset_name"]

    # Subsample
    rng = np.random.default_rng(42)
    idx = rng.choice(n_total, size=min(n_samples, n_total), replace=False)
    idx = np.sort(idx)
    sub_emb = embeddings[idx]
    sub_labels = asset_labels[idx]

    # t-SNE
    tsne = TSNE(
        n_components=2,
        perplexity=50,
        learning_rate=200,
        n_iter=1000,
        random_state=42,
    )
    coords = tsne.fit_transform(sub_emb)  # (n_samples, 2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("t-SNE of Regime Embeddings", fontsize=14, fontweight="bold")

    # Left: chronological order
    sc0 = axes[0].scatter(
        coords[:, 0], coords[:, 1],
        c=np.arange(len(coords)), cmap="viridis", s=4, alpha=0.7,
    )
    plt.colorbar(sc0, ax=axes[0], label="Window index (chronological)")
    axes[0].set_title("Temporal drift")
    axes[0].set_xlabel("t-SNE 1")
    axes[0].set_ylabel("t-SNE 2")

    # Right: by asset
    unique_assets = [a for a in _ASSET_COLORS if a in set(sub_labels)]
    for asset in unique_assets:
        mask = sub_labels == asset
        axes[1].scatter(
            coords[mask, 0], coords[mask, 1],
            c=_ASSET_COLORS[asset], label=asset, s=4, alpha=0.7,
        )
    axes[1].legend(markerscale=3, loc="upper right")
    axes[1].set_title("Asset membership")
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close(fig)

    mlflow.log_artifact(str(save_path))


def plot_training_curves(mlflow_run_id: str, save_path: Path) -> None:
    """
    4-subplot training dashboard loaded from MLflow metric history.
    Saves PNG to save_path and logs it as an MLflow artifact.
    """
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    client = mlflow.tracking.MlflowClient()

    def _get(metric_name: str) -> tuple[list[int], list[float]]:
        history = client.get_metric_history(mlflow_run_id, metric_name)
        steps = [m.step for m in history]
        values = [m.value for m in history]
        return steps, values

    train_steps, train_loss = _get("train_loss")
    val_steps, val_loss = _get("val_loss")
    _, train_acc = _get("train_top1_acc")
    _, val_acc = _get("val_top1_acc")
    temp_steps, temperature = _get("temperature")
    lr_steps, lr = _get("learning_rate")

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle("MARTA Encoder Training", fontsize=13, fontweight="bold")

    axes[0, 0].plot(train_steps, train_loss, label="train", color="#2196F3")
    axes[0, 0].plot(val_steps, val_loss, label="val", color="#F44336")
    axes[0, 0].set_title("InfoNCE Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(train_steps, train_acc, label="train", color="#2196F3")
    axes[0, 1].plot(val_steps, val_acc, label="val", color="#F44336")
    axes[0, 1].set_title("Top-1 Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(temp_steps, temperature, color="#9C27B0")
    axes[1, 0].axhline(0.07, linestyle="--", color="gray", linewidth=0.8, label="init=0.07")
    axes[1, 0].set_title("Learned Temperature")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(lr_steps, lr, color="#FF9800")
    axes[1, 1].set_title("Learning Rate")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close(fig)

    mlflow.log_artifact(str(save_path))


In [ ]:
%%writefile configs/encoder.yaml
model:
  input_dim: 14
  hidden_dim: 128
  proj_dim: 64
  n_lstm_layers: 2

training:
  window_size: 60
  step: 1
  batch_size: 512
  lr: 1.0e-3
  weight_decay: 1.0e-4
  max_epochs: 200
  early_stopping_patience: 20
  warmup_epochs: 10
  gradient_clip_max_norm: 1.0
  # 2026-07-13 walk-forward split: train 2018-01 -> 2023-12-31, val 2024.
  # Test (2025) is SACRED and not present in this dataset (marta-processed-data-v3
  # is trimmed to <= 2024-12-31 — see the leakage-guard cell above).
  train_end: "2024-01-01"
  val_start: "2024-01-01"

data:
  assets:
    - BTC_USDT
    - ETH_USDT
    - EURUSD
    - XAUUSD
  processed_dir: "data/processed"

visualization:
  tsne_n_samples: 3000
  tsne_perplexity: 50
  tsne_learning_rate: 200
  tsne_n_iter: 1000

mlflow:
  experiment_name: "marta_encoder_pretraining"
  run_name: "week2_encoder"

convergence:
  max_val_loss: 4.0
  min_top1_acc: 0.20
  min_temperature: 0.03
  max_temperature: 0.30

checkpoints:
  dir: "encoder/checkpoints"
  best: "best_encoder.pt"
  final: "final_encoder.pt"


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")

import yaml
import mlflow
from pathlib import Path
from encoder.train import train_encoder

# Load encoder-specific config
with open("configs/encoder.yaml") as f:
    config = yaml.safe_load(f)

# Read aligned parquets from the (read-only) Kaggle input dataset — DATA_DIR was
# auto-discovered above, don't hardcode the input slug here.
config["data"]["processed_dir"] = str(DATA_DIR)
# Write generated embeddings to a WRITABLE location (input mount is read-only).
# /kaggle/working/data/processed is also where visualize.py looks for the asset map.
config["data"]["output_dir"] = "/kaggle/working/data/processed"

# Start MLflow run and train
mlflow.set_experiment("marta_encoder_pretraining")

with mlflow.start_run(run_name="week2_contrastive_pretraining"):
    model = train_encoder(config)

print("Training complete. Check encoder/checkpoints/ for saved weights.")


In [ ]:
import os
from IPython.display import FileLink, display

# best_encoder.pt is the critical deliverable — bring it back to
# encoder/checkpoints/best_encoder.pt locally (back up the old one first).
for file_path in [
    "/kaggle/working/best_encoder.pt",
    "/kaggle/working/embeddings.npy",
    "/kaggle/working/embedding_timestamps.parquet",
    "/kaggle/working/tsne_embeddings.png",
]:
    if os.path.exists(file_path):
        display(FileLink(file_path))
    else:
        print(f"Not found (yet): {file_path}")

In [ ]:
import os
from IPython.display import FileLink, display

# best_encoder.pt is the critical deliverable — bring it back to
# encoder/checkpoints/best_encoder.pt locally (back up the old one first).
# Paths match where train_encoder() (cell above) actually wrote them:
#   - best_encoder.pt is saved directly under /kaggle/working/ (hardcoded in train.py)
#   - embeddings/timestamps go to config["data"]["output_dir"] = /kaggle/working/data/processed
#   - tsne plot goes to <_ROOT>/encoder/plots = /kaggle/working/encoder/plots
for file_path in [
    "/kaggle/working/best_encoder.pt",
    "/kaggle/working/data/processed/embeddings.npy",
    "/kaggle/working/data/processed/embedding_timestamps.parquet",
    "/kaggle/working/data/processed/embedding_asset_map.parquet",
    "/kaggle/working/encoder/plots/tsne_embeddings.png",
]:
    if os.path.exists(file_path):
        display(FileLink(file_path))
    else:
        print(f"Not found (yet): {file_path}")